In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document
import os 

In [4]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source" : "https://www.google.com"}
)

sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [4]:
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

# text_document = loader.load()

In [5]:
# PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader


# pdf_loader = PyPDFLoader(
#     "data/research2.pdf"
# )
# document = pdf_loader.load()

# Ingestion Pipeline

In [6]:
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print('total_pdfs:', num_docs)
    print('total pages:', len(all_docs))
    return all_docs

/var/folders/hl/6l0r2dr92lb6rtflm0nwv39r0000gn/T/ipykernel_7371/578667329.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [7]:
all_pdf_documents = load_all_pdfs()


total_pdfs: 1
total pages: 21


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [10]:
chunks = split_docs(all_pdf_documents)

len(chunks)

244

In [9]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        # print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)

    def generateEmbeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        # print(embeddings, "embeddings")
        return embeddings

In [10]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Vector Store

In [12]:
import os
import chromadb
import uuid

class VectorStoreManager:

    def __init__(
        self,
        persist_directory="data/vector_store",
        collection_name="pdf_doc"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store() 

    def _initialize_store(self):
        # Create folder if it doesn't exist
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create persistent Chroma client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Create or get collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "Vector store collection for PDF embeddings in RAG"
            }
        )

        # print(f"Initialized collection: {self.collection.name}")
        # print(f"Documents in collection: {self.collection.count()}")

    def add_documents(self, documents, embeddings):

            if len(documents) != len(embeddings):
                raise ValueError('Lenght of documents and embeddings are not matched')

            # store - IDs, document, embedding, metadata 

            ids = []
            document_content = []
            embedding_list = []
            all_metadata = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
               doc_id = f"doc_{uuid.uuid4()}"
               ids.append(doc_id)

               metadata = dict(doc.metadata)
               metadata["doc_index"] = i
               metadata["content_length"] = len(doc.page_content)
               all_metadata.append(metadata)

               document_content.append(doc.page_content)
               embedding_list.append(embedding.tolist())

               self.collection.add(
               ids=ids,
               metadatas=all_metadata,
               documents=document_content,
               embeddings=embedding_list
               )

            # print(f"Total added collection in vector db:",len(document_content))
            # print(f"Documents in collection: {self.collection.count()}")

In [13]:
vector_store = VectorStoreManager()

Initialized collection: pdf_doc
Documents in collection: 244


In [15]:
#documents -> chunks -> embeddings 

text = [doc.page_content for doc in chunks]

embedddings = embedding_manager.generateEmbeddings(text)
vector_store.add_documents(chunks, embedddings)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

[[-0.06483068 -0.07701042 -0.00197702 ...  0.05516809 -0.03390041
   0.0142332 ]
 [-0.05468737 -0.04878929 -0.06856583 ... -0.01211448 -0.00324052
   0.11706021]
 [-0.06524609 -0.02504016 -0.04540539 ... -0.05117238  0.03061165
   0.03897084]
 ...
 [-0.04638281 -0.07808168 -0.03579666 ...  0.04963212 -0.08682777
  -0.06420169]
 [ 0.0201383  -0.04678576  0.0204688  ...  0.07905411  0.00145207
   0.03949948]
 [-0.06597348  0.0049125  -0.02753692 ... -0.02634129 -0.00415467
  -0.00835987]] embeddings
Total added collection in vector db: 244
Documents in collection: 244


# Retrieval Pipeline

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class RAGRetrieval:

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieval(self, query, topK=5, score_threshold=0.0):
        # get query embeddings
        query_embeddings = self.embedding_manager.generateEmbeddings([query])[0]

        #Sematic Search

        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = topK
        )

        # cosine similarity

        retrieval_docs = []

        if results['documents'] and results['documents'][0]:

            ids = results['ids'][0]
            documents = results['documents'][0]
            distances = results['distances'][0]
            metadatas=results['metadatas'][0]

            for i, (doc_ids, document, distance, metadata) in enumerate(zip(ids, documents,distances, metadatas)):

                similarity_score = 1 - distance

                if similarity_score >= score_threshold:

                    retrieval_docs.append({
                        "id":ids,
                        "document":document,
                        "metadata":metadata,
                    "distance":metadata,
                    "similarity_score":similarity_score,
                    "rank": i +1 
                    
                    })
                    # print(f"total docs retrieval {len(retrieval_docs)}")

                else:
                    print("Data not Found!")
                    
                
        return retrieval_docs
        

In [47]:
rag_retrieval = RAGRetrieval(embedding_manager, vector_store)
rag_retrieval.retrieval("what is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-6.95707425e-02  9.52000096e-02  1.60214026e-02  6.80146739e-03
  -8.84049833e-02  1.42048327e-02  5.40277213e-02  4.56368662e-02
  -3.19217704e-02 -2.95637287e-02  5.67035889e-03  6.36049826e-03
   2.30016615e-02 -5.11627272e-02 -4.39477116e-02  5.89560904e-02
   5.50739653e-02  9.31472704e-02 -1.99182890e-02  5.56544866e-03
  -6.06279075e-02  7.54946396e-02 -1.02014292e-03 -3.40628661e-02
   4.00799140e-02  4.76241186e-02 -3.31660844e-02 -7.22286117e-04
   5.98663203e-02 -8.33193865e-03 -9.59183462e-03  5.34926802e-02
  -2.39231475e-02 -2.01412104e-02 -4.85539138e-02  1.87074672e-02
  -2.20365375e-02  6.23629577e-02  4.00254130e-03  2.97090337e-02
   1.27505036e-02  1.79135951e-03 -2.17953399e-02 -8.11001211e-02
  -2.61447509e-04  3.26574594e-02 -1.05920052e-02  3.05196345e-02
   3.11776511e-02 -1.16895232e-02  5.91249159e-03  5.38987713e-03
   2.91767605e-02  5.12313880e-02 -3.31499167e-02 -9.99479070e-02
   7.15584978e-02 -6.35875314e-02 -3.19283679e-02  2.67297719e-02
  -4.80685

[{'id': ['doc_7a73e02a-14ff-47a9-98a8-754562afa413',
   'doc_f90d3b10-a057-418c-adb3-0a557b2ad653',
   'doc_8774b31b-0faa-41e0-902c-f8c2a563877a',
   'doc_43015f93-813c-434a-9a22-0b9c3e16d2e1',
   'doc_1c3e1456-a8bb-4594-82b5-3ce039d68ddd'],
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'author': '',
   'page': 0,
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'source': 'data/pdfs/research2.pdf',
   'page_label': '1',
   'keywords': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'total_pages': 21,
   'doc_index': 11,
   'content_length': 288,
   'moddate': '2024-03-28T00:54:45+00:00',
   'producer': 

### Groq

In [58]:
# pip install langchain-groq
# !pip install langchain-openai

In [43]:
# from langchain_groq import ChatGroq 
from langchain_openai import ChatOpenAI

llm = ChatGroq(
    openai_api_key='',
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)


# def generate_output(query, rag_retrieval, llm, top_k=3):

#     # Retrieve documents
#     result = rag_retrieval.retrieval(query, top_k)

#     # Create context
#     context = (
#         "\n".join([doc["document"] for doc in result])
#         if result
#         else "No data found!"
#     )

#     prompt = f"""
# Use the given context to generate the answer for the query.

# Context:
# {context}

# Query:
# {query}

# Answer:
# """

#     response = llm.invoke(prompt)

#     # print(response.content)
#     return response.content

def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content


In [60]:
answer = generate_output("What is RAG?",rag_retrieval, llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-6.95707425e-02  9.52000096e-02  1.60214026e-02  6.80146739e-03
  -8.84049833e-02  1.42048327e-02  5.40277213e-02  4.56368662e-02
  -3.19217704e-02 -2.95637287e-02  5.67035889e-03  6.36049826e-03
   2.30016615e-02 -5.11627272e-02 -4.39477116e-02  5.89560904e-02
   5.50739653e-02  9.31472704e-02 -1.99182890e-02  5.56544866e-03
  -6.06279075e-02  7.54946396e-02 -1.02014292e-03 -3.40628661e-02
   4.00799140e-02  4.76241186e-02 -3.31660844e-02 -7.22286117e-04
   5.98663203e-02 -8.33193865e-03 -9.59183462e-03  5.34926802e-02
  -2.39231475e-02 -2.01412104e-02 -4.85539138e-02  1.87074672e-02
  -2.20365375e-02  6.23629577e-02  4.00254130e-03  2.97090337e-02
   1.27505036e-02  1.79135951e-03 -2.17953399e-02 -8.11001211e-02
  -2.61447509e-04  3.26574594e-02 -1.05920052e-02  3.05196345e-02
   3.11776511e-02 -1.16895232e-02  5.91249159e-03  5.38987713e-03
   2.91767605e-02  5.12313880e-02 -3.31499167e-02 -9.99479070e-02
   7.15584978e-02 -6.35875314e-02 -3.19283679e-02  2.67297719e-02
  -4.80685